# PadelVision — 06 Events: hits, bounces, rallies (stage 5, Kaggle)

Runs the full pipeline on a video and then detects **events** from the ball's path:
- **hit**: a sharp turn where the ball is at a player, checked by depth. The ball's size must fit that player's
  distance from the camera, so a far-court ball seen "inside" a near player's box isn't counted.
- **bounce**: the ball comes down the image and goes back up. Bounces get a court position, since the ball is on
  the ground there.
- **wall**: any other sharp turn (glass rebounds, net cord, hits we can't attribute).

It also groups ball activity into **rallies**. Every event is exported as an image crop with the ball's path drawn,
so the detections can be checked by eye. Stages 3–5 can be re-run locally from the zip
(`padelvision restats`, `ball-retrack`, `events`).

**Setup (Kaggle):** GPU T4, Internet on, video attached. **Run All** takes about 8–12 min.

In [ ]:
import os, subprocess, sys
ON_KAGGLE = os.path.exists("/kaggle")
ON_COLAB = "google.colab" in sys.modules
WORK = "/kaggle/working" if ON_KAGGLE else "/content"
print(f"Kaggle={ON_KAGGLE} Colab={ON_COLAB} workdir={WORK}")
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "No GPU: enable one in the notebook settings"

In [ ]:
REPO_URL = "https://github.com/Duggineniakhil/PadelVision.git"
BRANCH = "v2-rebuild"
REPO = f"{WORK}/PadelVision"

if os.path.exists(REPO):
    !git -C {REPO} pull --ff-only
else:
    !git clone --depth 1 -b {BRANCH} {REPO_URL} {REPO}
!pip install -q -e "{REPO}/pipeline[ml]"

# Editable installs aren't picked up by an already-running kernel; make the package importable now.
sys.path.insert(0, f"{REPO}/pipeline/src")
import padelvision
print("padelvision", padelvision.__version__)

In [ ]:
!cd {REPO} && python scripts/fetch_models.py

In [ ]:
from pathlib import Path

if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

# Leave empty to use the first video found under /kaggle/input (Kaggle) or MyDrive/padel (Colab).
VIDEO_PATH = ""

if not VIDEO_PATH:
    search_root = Path("/kaggle/input") if ON_KAGGLE else Path("/content/drive/MyDrive/padel")
    videos = sorted(p for ext in ("*.mp4", "*.mov", "*.avi", "*.mkv") for p in search_root.rglob(ext))
    print("Videos found:", *videos, sep="\n  ")
    assert videos, f"No video found under {search_root}; attach one or set VIDEO_PATH"
    VIDEO_PATH = str(videos[0])

assert os.path.exists(VIDEO_PATH), f"Not found: {VIDEO_PATH}"
print("Using video:", VIDEO_PATH)

In [ ]:
# court.json: pasted below, else ml/calibrations/<video name>.json in the repo, else court.json next to the video.
COURT_JSON_TEXT = """
"""

court_files = sorted(Path("/kaggle/input" if ON_KAGGLE else "/content/drive/MyDrive/padel").rglob("court.json"))
# Calibrations committed to the repo, named after the video file (e.g. ml/calibrations/Test_video.json).
repo_cal = Path(REPO) / "ml" / "calibrations" / f"{Path(VIDEO_PATH).stem}.json"
if repo_cal.exists():
    court_files.insert(0, repo_cal)
COURT_JSON = f"{WORK}/court.json"
if COURT_JSON_TEXT.strip():
    Path(COURT_JSON).write_text(COURT_JSON_TEXT)
elif court_files:
    Path(COURT_JSON).write_text(court_files[0].read_text())
    print("Using", court_files[0])
else:
    raise FileNotFoundError("No court.json: attach it to the dataset or paste it above")

from padelvision.court import CourtCalibration
cal = CourtCalibration.load(COURT_JSON)
print(f"court error {cal.reprojection_error_m()*100:.0f} cm, lens k1={cal.lens.k1:.3f} k2={cal.lens.k2:.3f}")

## 1. Players, ball, events

In [ ]:
RUN_DIR = f"{WORK}/runs/match1"
MAX_FRAMES = None
extra = f"--max-frames {MAX_FRAMES}" if MAX_FRAMES else ""
!cd {REPO} && python -m padelvision.cli analyze "{VIDEO_PATH}" --court {COURT_JSON} --out {RUN_DIR} {extra}
!cd {REPO} && python -m padelvision.cli ball-track "{VIDEO_PATH}" --run {RUN_DIR} {extra}
!cd {REPO} && python -m padelvision.cli events {RUN_DIR}

## 2. Summary

In [ ]:
import json
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

summary = json.loads(Path(f"{RUN_DIR}/rallies.json").read_text())
ev = pd.read_parquet(f"{RUN_DIR}/events.parquet")
ball = pd.read_parquet(f"{RUN_DIR}/ball.parquet")
print({k: v for k, v in summary.items() if k != "rallies"})
display(pd.DataFrame(summary["rallies"]))

fig, ax = plt.subplots(1, 2, figsize=(14, 4))
ok = ball["size"].notna()
ax[0].scatter(ball.v[ok], ball["size"][ok], s=2, alpha=0.3)
ax[0].set_xlabel("ball v (px, down = nearer the camera)"); ax[0].set_ylabel("ball size (px)")
ax[0].set_title("Ball size vs image height (depth cue)")
b = ev[ev.kind == "bounce"]
ax[1].scatter(b.x_m, b.y_m, c=np.where(b.in_court.fillna(False).astype(bool), "tab:green", "tab:red"))
ax[1].add_patch(plt.Rectangle((-5, -10), 10, 20, fill=False)); ax[1].axhline(0, color="k", lw=0.5)
for y in (-6.95, 6.95):
    ax[1].plot([-5, 5], [y, y], "k", lw=0.5)
ax[1].set_xlim(-7, 7); ax[1].set_ylim(-12, 12); ax[1].set_aspect("equal")
ax[1].set_title("Floor bounces on the court (camera side at the bottom)")
plt.tight_layout(); plt.show()

## 3. Event crops for review

In [ ]:
# One crop per event with the ball's path (+/- 10 frames) drawn, for checking by eye.
OUT = Path(f"{WORK}/events_review")
OUT.mkdir(parents=True, exist_ok=True)
R, TRAIL = 110, 10
need = set(ev.frame)
FR, cap, f = {}, cv2.VideoCapture(VIDEO_PATH), 0
while need and f <= max(need):
    ok, img = cap.read()
    if not ok:
        break
    if f in need:
        FR[f] = img
    f += 1
cap.release()
bt = ball.set_index("frame")
colors = {"hit": (0, 0, 255), "bounce": (0, 200, 0), "wall": (255, 120, 0)}
for kind in ["hit", "bounce", "wall"]:
    tiles = []
    for k, r in enumerate(ev[ev.kind == kind].itertuples()):
        img = FR[int(r.frame)].copy()
        path = bt.loc[[x for x in range(int(r.frame) - TRAIL, int(r.frame) + TRAIL + 1) if x in bt.index]]
        pts = path[["u", "v"]].to_numpy().round().astype(np.int32)
        if len(pts) > 1:
            cv2.polylines(img, [pts], False, (0, 255, 255), 1, cv2.LINE_AA)
        cv2.circle(img, (round(r.u), round(r.v)), 9, colors[kind], 2, cv2.LINE_AA)
        u, v = int(r.u), int(r.v)
        crop = cv2.copyMakeBorder(img, R, R, R, R, cv2.BORDER_CONSTANT)[v:v + 2 * R, u:u + 2 * R]
        label = f"{kind[0].upper()}{k} {r.t:.1f}s" + (f" P{int(r.player)}" if r.player == r.player else "")
        cv2.rectangle(crop, (0, 0), (2 * R, 18), (0, 0, 0), -1)
        cv2.putText(crop, label, (3, 13), cv2.FONT_HERSHEY_SIMPLEX, 0.42, (255, 255, 255), 1, cv2.LINE_AA)
        tiles.append(crop)
    for s in range(0, len(tiles), 48):
        ch = tiles[s:s + 48]
        while len(ch) % 8:
            ch.append(np.zeros_like(tiles[0]))
        sheet = np.vstack([np.hstack(ch[i:i + 8]) for i in range(0, len(ch), 8)])
        cv2.imwrite(str(OUT / f"{kind}_{s // 48}.jpg"), sheet, [cv2.IMWRITE_JPEG_QUALITY, 85])
    print(kind, len(tiles), "crops")

## 4. Rally review (wide context)

Full frames for every event of suspicious rallies (one player getting most of the hits), with all player
boxes and the ball's path, plus the event sequence and a short clip. Use it to tell a real point from a
drill, ball bouncing before a serve, or the tracker following a second ball. Download `rally_review.zip`.

In [ ]:
# Wide-context review of suspicious rallies: full frame per event (all player boxes + ids, ball path),
# the event sequence, and a short half-res clip per rally. Rallies where one player has >= 70% of the
# hits are picked automatically; set RALLY_IDS to choose by hand (0-based index into rallies.json).
import json, shutil
import cv2
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import FileLink

RALLY_IDS = None          # e.g. [4, 7]; None = auto
CLIP_SECONDS = 12         # per rally clip, from the rally start (None = whole rally)
TRAIL = 15                # ball path frames on each side of an event
SCALE = 0.5               # tile / clip scale

summary = json.loads(Path(f"{RUN_DIR}/rallies.json").read_text())
rallies = summary["rallies"]
ev = pd.read_parquet(f"{RUN_DIR}/events.parquet")
ball = pd.read_parquet(f"{RUN_DIR}/ball.parquet")
players = pd.read_parquet(f"{RUN_DIR}/players.parquet")

if RALLY_IDS is None:
    RALLY_IDS = [i for i, r in enumerate(rallies)
                 if r["hits"] >= 5 and max(r["hits_by_player"].values()) >= 0.7 * r["hits"]]
print("Reviewing rallies:", RALLY_IDS)

ROUT = Path(f"{WORK}/rally_review")
shutil.rmtree(ROUT, ignore_errors=True)
ROUT.mkdir(parents=True)
bt = ball.set_index("frame")
pf = {f: g for f, g in players.groupby("frame")}
ev_colors = {"hit": (0, 0, 255), "bounce": (0, 200, 0), "wall": (255, 120, 0)}
team_col = {"near": (255, 200, 0), "far": (200, 0, 255)}


def draw(img, f, event=None):
    for p in pf.get(f, pd.DataFrame()).itertuples():
        c = team_col.get(p.team, (200, 200, 200))
        cv2.rectangle(img, (int(p.x1), int(p.y1)), (int(p.x2), int(p.y2)), c, 2)
        cv2.putText(img, f"P{int(p.player)}", (int(p.x1), int(p.y1) - 5),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.7, c, 2, cv2.LINE_AA)
    path = bt.loc[[x for x in range(f - TRAIL, f + TRAIL + 1) if x in bt.index]]
    pts = path[["u", "v"]].dropna().to_numpy().round().astype(np.int32)
    if len(pts) > 1:
        cv2.polylines(img, [pts], False, (0, 255, 255), 2, cv2.LINE_AA)
    if f in bt.index and bt.loc[f, "u"] == bt.loc[f, "u"]:
        cv2.circle(img, (round(bt.loc[f, "u"]), round(bt.loc[f, "v"])), 4, (0, 255, 255), -1)
    if event is not None:
        cv2.circle(img, (round(event.u), round(event.v)), 14, ev_colors[event.kind], 3, cv2.LINE_AA)
    return img


cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)
for ri in RALLY_IDS:
    r = rallies[ri]
    s, e = r["start_frame"], r["end_frame"]
    rev = ev[(ev.frame >= s) & (ev.frame <= e)].sort_values("frame")
    seq = " ".join(f"{k[0].upper()}{'' if p != p else int(p)}" for k, p in zip(rev.kind, rev.player))
    print(f"\nRally {ri}: frames {s}-{e} ({(e - s) / fps:.1f} s), hits {r['hits_by_player']}\n  {seq}")

    # Event sheet: one full (scaled) frame per event, 3 per row.
    tiles = []
    for k, x in enumerate(rev.itertuples()):
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(x.frame))
        ok, img = cap.read()
        if not ok:
            continue
        img = draw(img, int(x.frame), x)
        lab = f"#{k} {x.kind} f{int(x.frame)} {x.t:.1f}s" + (f" P{int(x.player)}" if x.player == x.player else "")
        lab += f" size {x.size:.1f}px" if x.size == x.size else ""
        cv2.rectangle(img, (0, 0), (img.shape[1], 40), (0, 0, 0), -1)
        cv2.putText(img, lab, (8, 30), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2, cv2.LINE_AA)
        tiles.append(cv2.resize(img, None, fx=SCALE, fy=SCALE, interpolation=cv2.INTER_AREA))
    for s0 in range(0, len(tiles), 12):
        ch = tiles[s0:s0 + 12]
        while len(ch) % 3:
            ch.append(np.zeros_like(tiles[0]))
        sheet = np.vstack([np.hstack(ch[i:i + 3]) for i in range(0, len(ch), 3)])
        cv2.imwrite(str(ROUT / f"rally{ri}_events_{s0 // 12}.jpg"), sheet, [cv2.IMWRITE_JPEG_QUALITY, 85])

    # Short clip with the same overlays (events stay marked for 5 frames).
    end = e if CLIP_SECONDS is None else min(e, s + int(CLIP_SECONDS * fps))
    cap.set(cv2.CAP_PROP_POS_FRAMES, s)
    ok, img = cap.read()
    h, w = int(img.shape[0] * SCALE), int(img.shape[1] * SCALE)
    raw = ROUT / f"rally{ri}_raw.mp4"
    vw = cv2.VideoWriter(str(raw), cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
    evf = {int(x.frame): x for x in rev.itertuples()}
    f = s
    while ok and f <= end:
        near = next((evf[g] for g in range(f - 5, f + 1) if g in evf), None)
        img = draw(img, f, near)
        cv2.putText(img, f"f{f} {f / fps:.1f}s", (8, 30), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 2)
        vw.write(cv2.resize(img, (w, h), interpolation=cv2.INTER_AREA))
        ok, img = cap.read()
        f += 1
    vw.release()
    !ffmpeg -loglevel error -y -i {raw} -c:v libx264 -pix_fmt yuv420p -crf 26 {ROUT}/rally{ri}.mp4 && rm {raw}
cap.release()

ZIP = f"{WORK}/rally_review.zip"
shutil.make_archive(ZIP[:-4], "zip", ROUT.parent, ROUT.name)
print(f"\n{ZIP}: {sorted(p.name for p in ROUT.iterdir())}")
FileLink(Path(ZIP).relative_to(Path.cwd()))

## 5. Hit/bounce labelling pack

Every frame of the windows to label by hand (all rallies, other activity with hits, 3 random windows;
at most 2 min). Download `event_review.zip`, unzip it into `data/` and run
`padelvision label-events data/event_review` locally. The labels give hit/bounce precision, recall and F1
(`padelvision events-eval`).

In [ ]:
import shutil
from pathlib import Path
from IPython.display import FileLink

EVENT_PACK = f"{WORK}/event_review"
shutil.rmtree(EVENT_PACK, ignore_errors=True)
!cd {REPO} && python -m padelvision.cli event-review-pack "{VIDEO_PATH}" --run {RUN_DIR} --out {EVENT_PACK}
EZIP = shutil.make_archive(EVENT_PACK, "zip", WORK, "event_review")
print(f"{EZIP}: {Path(EZIP).stat().st_size / 1e6:.0f} MB")
FileLink(Path(EZIP).relative_to(Path.cwd()))

## 6. Download

In [ ]:
import shutil
from IPython.display import FileLink
for name in ["events.parquet", "rallies.json", "ball.parquet", "ball_stats.json", "ball_detections.parquet",
             "ball_detections.json", "players.parquet", "detections.parquet", "detections.json", "stats.json",
             "video.json", "court.json"]:
    p = Path(RUN_DIR) / name
    if p.exists():
        shutil.copy(p, OUT / name)
ZIP = f"{WORK}/events_review.zip"
shutil.make_archive(ZIP[:-4], "zip", OUT.parent, OUT.name)
print(f"{ZIP}: {Path(ZIP).stat().st_size / 1e6:.1f} MB")
FileLink(Path(ZIP).relative_to(Path.cwd()) if Path(ZIP).is_relative_to(Path.cwd()) else ZIP)